# Introdução ao Deploy de Modelos usando TensorFlowModel

## Visão Geral

A classe `TensorFlowModel` do Accelerated Data Science (ADS) foi projetada para permitir que você coloque rapidamente um modelo em produção.
O método `.prepare()` cria os artefatos do modelo necessários para realizar o deploy de um modelo funcional sem a necessidade de configurar ou escrever código adicional. No entanto, ele permite que você personalize o arquivo `score.py`, caso seja necessário.
É possível simular uma chamada a um modelo implantado utilizando o método `.verify()`. Esse método executa as funções `load_model()` e `predict()` definidas no arquivo `score.py`. O uso do `.verify()` permite depurar o arquivo `score.py` sem a necessidade de realizar o deploy do modelo.
O método `.save()` envia o seu `TensorFlowModel` e os artefatos do modelo para o **Model Catalog**.
O método `.deploy()` realiza o deploy do modelo em um endpoint REST.
Por fim, o método `.predict()` permite chamar o endpoint para realizar a inferência do modelo.
Esses passos simples permitem que você leve um modelo TensorFlow treinado para produção com apenas algumas linhas de código.


## Contents

- Introduction  
  - Dataset
- Create a TensorFlow Model
- TensorFlow Framework Serialization
  - Create a TensorFlowModel
  - Prepare
  - Verify
  - Save
  - Deploy
  - Predict
- Clean Up
- References

## Configuração do Dataset e do Ambiente

Os datasets são fornecidos apenas como uma conveniência. Os datasets são considerados conteúdo de terceiros e não são considerados materiais cobertos 
pelo seu contrato com a Oracle.
O dataset Fashion_MNIST é utilizado neste notebook.

In [ ]:
# Core and OCI ADS imports
import ads
import logging
import os
import pandas as pd
import tempfile
import tensorflow as tf
import tensorflow_datasets as tfds
import warnings

# OCI ADS Model Catalog and Dataset utilities
from ads.catalog.model import ModelCatalog
from ads.model.model_metadata import UseCaseType
from ads.dataset.factory import DatasetFactory

In [ ]:
# Authentication
ads.set_auth(auth='resource_principal')

# Introdução

## Dataset

O **Fashion_MNIST** é um conjunto de dados de imagens de artigos de vestuário da Zalando, consistindo em um conjunto de treinamento 
com **60.000 exemplos** e um conjunto de teste com **10.000 exemplos**. 
Cada exemplo é uma imagem em escala de cinza de **28x28 pixels**, associada a um rótulo de **10 classes**.

O Fashion-MNIST serve como um substituto direto do dataset original **MNIST**, sendo amplamente utilizado para benchmark de algoritmos de 
aprendizado de máquina. Ele compartilha o mesmo tamanho de imagens e a mesma estrutura de divisão entre treino e teste.

Cada imagem possui **28 pixels de altura e 28 pixels de largura**, totalizando **784 pixels**. 
Cada pixel possui um único valor associado, que indica o nível de luminosidade ou escuridão, onde valores maiores representam pixels mais escuros. 
O valor de cada pixel é um inteiro entre **0 e 255**.

Os conjuntos de treinamento e teste possuem **785 colunas**. 
A primeira coluna corresponde ao rótulo da classe (conforme descrito acima), e as demais representam os valores dos pixels da imagem associada.

> **Nota:** Este notebook requer acesso público à internet para realizar o download do dataset de exemplo.

In [ ]:
# Visualizando algumas imagens do conjunto de treino

from matplotlib import pyplot as plt

for i in range(9):
    plt.subplot(330 + 1 + i)
    plt.imshow(x_train[i], cmap='gray')

plt.show()

In [ ]:
class TFModel:

    fmnist = tf.keras.datasets.fashion_mnist
    (x_train, y_train), (x_test, y_test) = fmnist.load_data()  # Load data

    x_train, x_test = x_train / 255.0, x_test / 255.0         # Scale between 0 and 1
    x_train, y_train = x_train[:10000], y_train[:10000]       # Reduce training data size

    def training(self):
        model = tf.keras.models.Sequential(
            [
                tf.keras.layers.Flatten(input_shape=(28, 28)),
                tf.keras.layers.Dense(128, activation="relu"),
                tf.keras.layers.Dropout(0.2),
                tf.keras.layers.Dense(10),
            ]
        )

        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

        model.compile(
            optimizer="adam",
            loss=loss_fn,
            metrics=["accuracy"]
        )

        model.fit(self.x_train, self.y_train, epochs=1)

        return model

## Treinamento e Predição do Modelo

Após a definição da classe do modelo, a próxima célula instancia o modelo e executa o treinamento.
Depois que o modelo é treinado, utiliza-se o método `.predict()` para realizar predições em um subconjunto do dataset de teste.
Cada predição retorna **10 valores**, pois existem **10 nós na camada de saída**, um para cada classe possível.  
O valor mais alto indica a classe que o modelo prevê como correta.

In [ ]:
# Realizando predições em um subconjunto do dataset de teste
model.predict(TFModel().x_test[:1])

## Criação de um TensorFlowModel

Neste notebook, um objeto `keras.engine.sequential.Sequential` é criado.  
O construtor `TensorFlowModel()` recebe o objeto `keras.engine.sequential.Sequential` juntamente com o caminho onde serão armazenados os artefatos do modelo.

Um objeto `TensorFlowModel` é retornado e utilizado para gerenciar o processo de deploy do modelo.

A próxima célula cria um diretório de artefatos do modelo.  
Esse diretório é utilizado para armazenar os artefatos necessários para realizar o deploy do modelo e também cria o objeto `TensorFlowModel`.

In [ ]:
# Criando o objeto TensorFlowModel
tf_model = TensorFlowModel(estimator=model, artifact_dir=artifact_dir)

In [ ]:
# Exibindo o status do processo de deploy
tf_model.summary_status()

Para criar os artefatos do modelo, utiliza-se o método `.prepare()`.  
Existem diversos parâmetros que permitem armazenar informações de proveniência do modelo.

Na próxima célula, a variável `conda_env` define o ambiente Conda utilizado para treinar o modelo, e o ambiente Conda a ser utilizado no deploy.

Observe que é possível passar apenas *slugs* para `inference_conda_env` ou `training_conda_env` se for um ambiente de serviço.  
Caso contrário, é necessário informar o caminho completo do ambiente Conda juntamente com a versão do Python por meio de `inference_python_version` e `training_python_version`.

In [ ]:
# Preparação dos artefatos do modelo
tf_model.prepare(
    inference_conda_env=conda_env,
    training_conda_env=conda_env,
    use_case_type=UseCaseType.MULTINOMIAL_CLASSIFICATION,
    X_sample=TFModel().x_test,
    y_sample=TFModel().y_test,
)

In [ ]:
# Exibindo o status da etapa Prepare
tf_model.summary_status()

In [ ]:
# Listando os artefatos gerados
os.listdir(artifact_dir)

In [ ]:
tf_model.runtime_info

In [ ]:
tf_model.schema_input

In [ ]:
tf_model.metadata_custom

In [ ]:
tf_model.metadata_provenance

In [ ]:
tf_model.metadata_taxonomy

## Verify

Se você modificar o arquivo `score.py`, que faz parte dos artefatos do modelo, então deve executar a etapa de verificação.
A etapa **Verify** permite testar essas alterações sem a necessidade de realizar o deploy do modelo.  
Isso permite depurar o código sem precisar salvar o modelo no Model Catalog e, em seguida, implantá-lo.
O método `.verify()` recebe um conjunto de parâmetros de teste e realiza a predição chamando a função `predict()` definida no arquivo `score.py`.  
Ele também executa a função `load_model()`.

A próxima célula simula uma chamada a um modelo implantado, sem que seja necessário realizar o deploy de fato.  
Ela passa valores de teste e retorna as predições.

Atualize o método `.summary_status()` para mostrar que a etapa de verificação foi concluída.

## Save

Depois que você estiver satisfeito com a performance do modelo e tiver verificado que o arquivo `score.py` está funcionando corretamente, você pode salvar o modelo no **Model Catalog**.
Isso é feito utilizando o método `.save()` em um objeto `TensorFlowModel`.  
Esse método empacota o artefato do modelo que foi criado e o publica no Model Catalog.
Como resultado, o método retorna o **OCID do modelo**.

## Deploy

Quando o modelo está no Model Catalog, você pode utilizar o método `.deploy()` de um objeto `TensorFlowModel` para realizar o deploy do modelo.
Esse método permite especificar diversos atributos do deploy, tais como:
- nome de exibição (display name)
- descrição
- tipo e quantidade de instâncias
- largura de banda máxima
- grupos de logging

A próxima célula realiza o deploy do modelo utilizando as configurações padrão, com exceção do nome de exibição customizado.
O método `.deploy()` retorna um objeto do tipo `ModelDeployment`.

Após o deploy, o método `.summary_status()` mostra que o modelo está no estado **ACTIVE** e que o método `.predict()` está disponível.

## Predict

Na seção **Create a TensorFlow Model**, você utilizou o método `model.predict()`, onde `model` é um objeto do tipo `ADSModel`.  
Nesse caso, a inferência foi realizada utilizando o modelo local.
Agora que o modelo `TensorFlowModel` foi implantado, você pode realizar a mesma operação utilizando uma sintaxe semelhante com o método `.predict()` em um objeto `TensorFlowModel`.
Após o deploy estar ativo, você pode chamar o método `predict()` no objeto `TensorFlowModel` para enviar uma requisição ao endpoint implantado.

## Clean Up

Este notebook criou um deployment de modelo e um modelo.  
Esta seção remove esses recursos.
O deployment do modelo deve ser excluído antes que o modelo possa ser removido.  
Para isso, utiliza-se o método `.delete_deployment()` em um objeto `TensorFlowModel`.

Após o deployment do modelo ter sido excluído, o método `.summary_status()` mostra que o modelo foi removido e que o método `predict()` não está mais disponível.

In [ ]:
# Verificando o status após remover o deployment
tf_model.summary_status()

In [ ]:
# Excluindo o modelo do Model Catalog
ModelCatalog(
    compartment_id=os.environ["NB_SESSION_COMPARTMENT_OCID"]
).delete_model(model_id)

In [ ]:
# Removendo os artefatos locais do modelo
from shutil import rmtree
rmtree(artifact_dir)